In [ ]:
import pandas as pd
import numpy as np
from itertools import product
from joblib import Parallel, delayed
from typing import Dict, List, Any



class TrailingStopOptimizer:
    def __init__(self, df: pd.DataFrame, timeframe: str = '1h'):
        """
        Initialize the optimizer with OHLCV data.
        
        Parameters:
        -----------
        df : pd.DataFrame
            DataFrame containing 'Open', 'High', 'Low', 'Close'.
        timeframe : str
            Timeframe identifier (e.g., '1h', '4h'). Used for logging.
        """
        self.df = df.copy()
        self.timeframe = timeframe
        self.best_return = None
        self.best_params = {}

    def _run_strategy(self, params: Dict[str, Any]) -> pd.DataFrame:
        """
        Internal method to compute signals based on specific parameters.
        This encapsulates the logic from the previous prompt.
        """
        n = params.get('n', 14)
        factor_up = params.get('factor_up', 2.0)
        factor_down = params.get('factor_down', 2.0)
        profit_multiplier = params.get('profit_multiplier', 1.5)
        max_number_candles = params.get('max_number_candles', 20)

        # --- ATR Calculation ---
        tr1 = self.df['High'] - self.df['Low']
        tr2 = abs(self.df['High'] - self.df['Close'].shift(1))
        tr3 = abs(self.df['Low'] - self.df['Close'].shift(1))
        true_range = pd.concat([tr1, tr2, tr3], axis=1).max(axis=1)
        atr = true_range.rolling(window=n).mean()

        # --- Initialize Signals ---
        df_signals = self.df.copy()
        df_signals['trailing_signal_up'] = 0
        df_signals['trailing_signal_down'] = 0
        
        highs = df_signals['High'].values
        lows = df_signals['Low'].values
        closes = df_signals['Close'].values
        atrs = atr.values
        length = len(df_signals)
        
        # Limit loop to ensure we have enough future data for simulation
        limit_index = length - max_number_candles

        for i in range(limit_index):
            if np.isnan(atrs[i]):
                continue
                
            current_atr = atrs[i]
            entry_price = closes[i]
            
            # --- Long Logic ---
            long_stop_dist = current_atr * factor_up
            long_target = entry_price + (profit_multiplier * long_stop_dist)
            signal_up = 0
            current_trailing_stop_long = entry_price - long_stop_dist
            
            start_idx = i + 1
            end_idx = min(i + max_number_candles, length)
            
            # Optimization: Slice arrays once per candle to avoid repeated indexing overhead
            future_highs = highs[start_idx:end_idx]
            future_lows = lows[start_idx:end_idx]
            future_closes = closes[start_idx:end_idx]

            for j in range(len(future_highs)):
                high_j = future_highs[j]
                low_j = future_lows[j]
                
                if low_j <= current_trailing_stop_long:
                    break 
                
                max_high_so_far = np.max(future_highs[:j+1])
                new_stop = max_high_so_far - long_stop_dist
                
                if new_stop > current_trailing_stop_long:
                    current_trailing_stop_long = new_stop
                    
                if high_j >= long_target or future_closes[j] >= long_target:
                    signal_up = 1
                    break
            
            df_signals.at[i, 'trailing_signal_up'] = signal_up

            # --- Short Logic ---
            short_stop_dist = current_atr * factor_down
            short_target = entry_price - (profit_multiplier * short_stop_dist)
            signal_down = 0
            current_trailing_stop_short = entry_price + short_stop_dist
            
            for j in range(len(future_highs)):
                high_j = future_highs[j]
                low_j = future_lows[j]
                
                if high_j >= current_trailing_stop_short:
                    break
                
                min_low_so_far = np.min(future_lows[:j+1])
                new_stop = min_low_so_far + short_stop_dist
                
                if new_stop < current_trailing_stop_short:
                    current_trailing_stop_short = new_stop
                    
                if low_j <= short_target or future_closes[j] <= short_target:
                    signal_down = 1
                    break
            
            df_signals.at[i, 'trailing_signal_down'] = signal_down

        return df_signals

    def _calculate_score(self, df_signals: pd.DataFrame) -> float:
        """
        Calculate the performance metric for a given set of signals.
        Currently maximizes Total Profitable Signals (Win Count).
        """
        # Sum of successful long and short signals
        total_wins = df_signals['trailing_signal_up'].sum() + df_signals['trailing_signal_down'].sum()
        
        # Optional: You could add penalties for false positives or calculate Sharpe here
        return float(total_wins)

    def _evaluate_single_params(self, params: Dict[str, Any]) -> tuple:
        """
        Wrapper to run strategy and score it. Returns (score, params).
        Used by joblib parallel processing.
        """
        try:
            df_signals = self._run_strategy(params)
            score = self._calculate_score(df_signals)
            return (score, params)
        except Exception as e:
            print(f"Error evaluating {params}: {e}")
            return (-1, params)

    def _evaluate_parallel(self, param_dicts: List[Dict], n_jobs: int, verbose: bool) -> List[tuple]:
        """
        Evaluate all parameter combinations in parallel.
        """
        results = Parallel(n_jobs=n_jobs)(delayed(self._evaluate_single_params)(p) for p in param_dicts)
        
        if verbose:
            print(f"Completed {len(results)} evaluations.")
            
        return results

    def _create_result_dict(
        self, 
        method: str, 
        n_tested: int, 
        results: List[tuple], 
        timeframe: str
    ) -> Dict[str, Any]:
        """Format the output dictionary."""
        # Sort results by score descending for easier inspection
        sorted_results = sorted(results, key=lambda x: x[0], reverse=True)
        
        return {
            'method': method,
            'timeframe': timeframe,
            'n_tested': n_tested,
            'best_score': self.best_return,
            'best_params': self.best_params,
            'top_5_results': sorted_results[:5]
        }

    def grid_search(
        self,
        param_grid: Dict[str, List],
        n_jobs: int = -1,
        verbose: bool = True
    ) -> Dict:
        """Perform grid search optimization."""
        param_names = list(param_grid.keys())
        param_values = list(param_grid.values())
        
        combinations = list(product(*param_values))
        total_combinations = len(combinations)
        
        if verbose:
            print(f"Grid search ({self.timeframe}): Testing {total_combinations} combinations")
        
        param_dicts = [dict(zip(param_names, combo)) for combo in combinations]
        results = self._evaluate_parallel(param_dicts, n_jobs, verbose)
        
        # Find best result (max score)
        if not results:
            raise ValueError("No valid results returned from evaluation.")
            
        best_idx = np.argmax([r[0] for r in results])
        self.best_return, self.best_params = results[best_idx]
        
        return self._create_result_dict(
            method='grid_search',
            n_tested=total_combinations,
            results=results,
            timeframe=self.timeframe
        )

# ==========================================
# Example Usage
# ==========================================
if __name__ == "__main__":
    # 1. Generate Dummy Data
    np.random.seed(42)
    dates = pd.date_range(start='2023-01-01', periods=500, freq='D')
    df_data = pd.DataFrame({
        'Open': 100 + np.cumsum(np.random.randn(500)),
        'High': (100 + np.cumsum(np.random.randn(500))) * 1.02,
        'Low': (100 + np.cumsum(np.random.randn(500))) * 0.98,
        'Close': 100 + np.cumsum(np.random.randn(500))
    }, index=dates)

    # Fix OHLC validity
    df_data['High'] = df_data[['High', 'Open', 'Close']].max(axis=1)
    df_data['Low'] = df_data[['Low', 'Open', 'Close']].min(axis=1)

    # 2. Initialize Optimizer
    optimizer = TrailingStopOptimizer(df=df_data, timeframe='1D')

    # 3. Define Parameter Grid
    param_grid = {
        'n': [7, 14],                  # ATR Period
        'factor_up': [1.5, 2.0],       # Long Stop Distance Multiplier
        'factor_down': [1.5, 2.0],     # Short Stop Distance Multiplier
        'profit_multiplier': [1.0, 1.5], # Target Profit Multiplier
        'max_number_candles': [10, 20]   # Look-ahead window size
    }

    # 4. Run Grid Search
    # Note: n_jobs=-1 uses all available CPU cores
    results = optimizer.grid_search(
        param_grid=param_grid, 
        n_jobs=-1, 
        verbose=True
    )

    print("\n--- Optimization Results ---")
    print(f"Best Score (Total Wins): {results['best_score']}")
    print(f"Best Parameters: {results['best_params']}")
    
    # 5. Apply Best Params to Data
    best_signals_df = optimizer._run_strategy(optimizer.best_params)
    print("\nSignal Distribution:")
    print(best_signals_df[['trailing_signal_up', 'trailing_signal_down']].sum())
